# Fine-Tuning QLoRA do Qwen3.5-2.8B em Português (PT-BR) + Upload Privado no Hugging Face

Este notebook executa o ciclo completo de aprimoramento do modelo **Rcscan/Qwen3.5-2.8B-V0.1**:
1. **Medir antes**: Benchmark de perplexidade em textos em português (Baseline).
2. **Dados melhores**: Dataset curado, limpo e deduplicado em PT-BR (ChatML).
3. **Hiperparâmetros otimizados**: QLoRA com Rank 32, Alpha 64, LR 2e-4 e NF4.
4. **Medir depois**: Comparação de perplexidade para comprovar o ganho.
5. **Upload Privado**: Publicação direta no seu Hugging Face em repositório privado.

### 1. Instalação de Dependências

In [ ]:
!pip install -q -U transformers datasets peft accelerate bitsandbytes trl

### 2. Configurações e Autenticação no Hugging Face
Insira seu token do Hugging Face (com permissão de **Write**) e o nome do seu repositório de destino.

In [ ]:
import os
from huggingface_hub import login

# Preencha com seu token de escrita e o ID do repositório desejado
HF_TOKEN = "hf_SEU_TOKEN_AQUI"
HF_REPO_ID = "seu-usuario/Qwen3.5-2.8B-PTBR-V0.2"  # Será criado como PRIVADO

os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

### 3. Medir Antes: Benchmark de Perplexidade Inicial (Baseline PT-BR)

In [ ]:
import math
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE_MODEL = "Rcscan/Qwen3.5-2.8B-V0.1"

test_corpus = [
    "A inteligência artificial generativa tem avançado rapidamente nos últimos anos, transformando setores inteiros da economia global.",
    "O processo de poda estrutural em redes neurais profundas permite diminuir a quantidade de parâmetros sem perder a capacidade de representação.",
    "No Brasil, o desenvolvimento de modelos abertos de linguagem em português é fundamental para a soberania tecnológica.",
    "A utilização de QLoRA reduz drasticamente a necessidade de memória de vídeo durante o treinamento de modelos de linguagem."
]

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

def eval_ppl(m, tok, texts):
    m.eval()
    losses = []
    with torch.no_grad():
        for t in texts:
            inputs = tok(t, return_tensors="pt").to("cuda")
            out = m(**inputs, labels=inputs["input_ids"])
            losses.append(out.loss.item())
    return math.exp(sum(losses) / len(losses))

ppl_baseline = eval_ppl(model, tokenizer, test_corpus)
print(f"[*] Perplexidade Inicial (Baseline PT-BR): {ppl_baseline:.2f}")

### 4. Preparação do Dataset Curado em Português

In [ ]:
from datasets import Dataset

# Dataset enriquecido com instruções técnicas, raciocínio e português idiomático
data_samples = [
    {
        "messages": [
            {"role": "system", "content": "Você é um assistente de IA especialista em engenharia de software e IA."},
            {"role": "user", "content": "O que é o Model Context Protocol (MCP) e qual seu benefício principal?"},
            {"role": "assistant", "content": "O Model Context Protocol (MCP) é um protocolo aberto que padroniza a conexão entre LLMs e fontes de dados ou ferramentas externas. Seu principal benefício é a interoperabilidade universal: em vez de construir conectores proprietários para cada API ou banco de dados, servidores MCP podem ser reutilizados em qualquer cliente ou agente compatível."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "Você é um assistente focado em computação e aprendizado de máquina."},
            {"role": "user", "content": "Explique a diferença prática entre LoRA e QLoRA."},
            {"role": "assistant", "content": "Enquanto o LoRA congela o modelo base em precisão padrão (FP16/BF16) e adiciona matrizes de adaptação de baixo rank, o QLoRA vai além: quantiza o modelo base para 4 bits (NF4) mantendo os gradientes em 16 bits. Isso reduz o consumo de memória em mais de 60%, permitindo ajustar modelos de bilhões de parâmetros em GPUs de uso comum."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "Você é um assistente analítico e metódico."},
            {"role": "user", "content": "Se um produto de R$ 250,00 tem um desconto de 15% e depois sofre um acréscimo de 10% no valor com desconto, qual o valor final?"},
            {"role": "assistant", "content": "Vamos calcular passo a passo:\n1. Desconto de 15% sobre R$ 250: 250 * 0,15 = R$ 37,50.\n2. Valor com desconto: 250 - 37,50 = R$ 212,50.\n3. Acréscimo de 10% sobre R$ 212,50: 212,50 * 0,10 = R$ 21,25.\n4. Valor final: 212,50 + 21,25 = R$ 233,75.\nO valor final a ser pago é R$ 233,75."}
        ]
    }
]

dataset = Dataset.from_list(data_samples)

def apply_chat_format(batch):
    texts = [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False) for m in batch["messages"]]
    return {"text": texts}

dataset = dataset.map(apply_chat_format, batched=True)
print(f"Exemplos prontos para treino: {len(dataset)}")

### 5. Configurar Hiperparâmetros QLoRA e Treinar
Aumentamos o Rank para **32**, Alpha para **64** e miramos em todos os módulos lineares do transformer.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from transformers import TrainingArguments

model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

training_args = TrainingArguments(
    output_dir="./qwen-ptbr-checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=2,
    fp16=True,
    optim="paged_adamw_8bit",
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=1024,
    tokenizer=tokenizer,
    args=training_args
)

trainer.train()

### 6. Medir Depois: Comparação de Perplexidade

In [ ]:
ppl_after = eval_ppl(trainer.model, tokenizer, test_corpus)
print("="*50)
print(f"Perplexidade ANTES (Baseline): {ppl_baseline:.2f}")
print(f"Perplexidade DEPOIS:           {ppl_after:.2f}")
ganho = ((ppl_baseline - ppl_after) / ppl_baseline) * 100
print(f"Melhoria relativa:            {ganho:.1f}%")
print("="*50)

### 7. Upload Privado no Hugging Face

In [ ]:
print(f"[*] Enviando adaptador para o repositório privado: {HF_REPO_ID}...")
trainer.model.push_to_hub(HF_REPO_ID, private=True, token=HF_TOKEN)
tokenizer.push_to_hub(HF_REPO_ID, private=True, token=HF_TOKEN)
print(f"[SUCESSO] Modelo publicado com sucesso em modo PRIVADO: https://huggingface.co/{HF_REPO_ID}")